# 📖 Notebook 1: Geolocation-Based Matching

When you open Tinder, you see profiles of people **near you**. Behind this is a geospatial query — given your GPS coordinates, find all users within X kilometers who also match your preferences (age, gender).

Regular database indexes (B-trees) are terrible at this because latitude and longitude are **two dimensions**. A B-tree can efficiently search one column, but searching two at once (lat AND lng) requires scanning far too many rows.

This notebook shows how to solve this with **PostGIS** — a PostgreSQL extension that adds spatial indexes (R-trees) purpose-built for geographic queries.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why regular SQL queries fail for location-based matching
- How PostGIS uses spatial indexes to find nearby users efficiently
- How to combine location queries with preference filters (age, gender)
- How to exclude already-swiped users from the feed
- How to measure the performance difference between naive and spatial queries

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/tinder
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `tinder_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tinder_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    """Create a new database connection."""
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    """Run a query and return all rows as dictionaries."""
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

# Test connection
try:
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("SELECT PostGIS_Version();")
    version = cursor.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL + PostGIS (version {version})")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

## 👀 Let's See Our Users

Our database is pre-loaded with 20 sample users scattered around **Los Angeles**.  
Each user has a name, age, gender, preferences, and a GPS location.

Let's take a look at who's on our platform:

In [ ]:
# See all users and their locations
users = query("""
    SELECT 
        id, name, age, gender, interested_in,
        age_min, age_max, max_distance_km,
        ST_X(location::geometry) AS longitude,
        ST_Y(location::geometry) AS latitude
    FROM users
    ORDER BY id
""")

print(f"📊 {len(users)} users on the platform\n")
print(f"{'ID':<4} {'Name':<20} {'Age':<5} {'Gender':<8} {'Seeks':<8} {'Lat':>8} {'Lng':>10}")
print("-" * 75)
for u in users:
    print(f"{u['id']:<4} {u['name']:<20} {u['age']:<5} {u['gender']:<8} {u['interested_in']:<8} {u['latitude']:>8.4f} {u['longitude']:>10.4f}")

## 🤔 The Naive Approach: Bounding Box Query

The simplest way to find nearby users is a **bounding box** — filter by a latitude and longitude range:

```sql
SELECT * FROM users
WHERE lat BETWEEN my_lat - delta AND my_lat + delta
AND lng BETWEEN my_lng - delta AND my_lng + delta
```

### Why this is bad:
1. **Not circular** — a bounding box includes corners that are farther away than the radius
2. **Distortion** — 1 degree of longitude is ~111 km at the equator but ~85 km in LA. The box is a rectangle, not a square.
3. **No real distance** — you can't sort by "closest first" easily
4. **Index limitations** — a B-tree on (lat, lng) can only use one dimension efficiently

Let's try it anyway to see the problem:

In [ ]:
# Naive bounding box approach
# Let's find users near Downtown LA (-118.2437, 34.0522) within ~10 km
# 1 degree latitude ≈ 111 km, so 10 km ≈ 0.09 degrees
# 1 degree longitude at LA latitude ≈ 92 km, so 10 km ≈ 0.109 degrees

my_lat = 34.0522   # Downtown LA
my_lng = -118.2437
delta_lat = 0.09    # ~10 km in latitude
delta_lng = 0.109   # ~10 km in longitude

naive_results = query("""
    SELECT id, name, age, gender,
           ST_Y(location::geometry) AS latitude,
           ST_X(location::geometry) AS longitude
    FROM users
    WHERE ST_Y(location::geometry) BETWEEN %s AND %s
    AND ST_X(location::geometry) BETWEEN %s AND %s
    AND id != 1
""", (my_lat - delta_lat, my_lat + delta_lat, my_lng - delta_lng, my_lng + delta_lng))

print(f"📦 Naive bounding box found {len(naive_results)} users near Downtown LA:\n")
for u in naive_results:
    print(f"  {u['name']:<20} age={u['age']}  gender={u['gender']}")

print()
print("⚠️  Problems with this approach:")
print("   - No actual distance calculation")
print("   - Box shape includes corners that are > 10km away")
print("   - Can't easily sort by distance")
print("   - Doesn't account for Earth's curvature")

## ✅ The Right Approach: PostGIS Spatial Queries

PostGIS adds a `GEOGRAPHY` data type that understands **points on Earth's surface**. It uses a spatial index (called a GIST index) that efficiently searches in two dimensions.

The key function is **`ST_DWithin(geog1, geog2, distance_meters)`** — it returns TRUE if two geographic points are within the given distance of each other. PostGIS uses the spatial index to quickly eliminate far-away points without scanning every row.

Think of it like a library: instead of checking every book, the spatial index organizes space into a tree of bounding boxes. It quickly narrows down to a small region, then checks individual points.

In [ ]:
# PostGIS approach: find users within 10 km of Downtown LA
# ST_DWithin uses meters, so 10 km = 10000 m

my_location = "ST_SetSRID(ST_MakePoint(-118.2437, 34.0522), 4326)"

postgis_results = query(f"""
    SELECT 
        id, name, age, gender,
        ROUND(ST_Distance(location, {my_location})::numeric, 0) AS distance_meters
    FROM users
    WHERE ST_DWithin(location, {my_location}, 10000)  -- 10 km in meters
    AND id != 1  -- exclude ourselves
    ORDER BY ST_Distance(location, {my_location})
""")

print(f"🎯 PostGIS found {len(postgis_results)} users within 10 km of Downtown LA:\n")
print(f"{'Name':<20} {'Age':<5} {'Gender':<8} {'Distance':>10}")
print("-" * 48)
for u in postgis_results:
    dist_km = float(u['distance_meters']) / 1000
    print(f"{u['name']:<20} {u['age']:<5} {u['gender']:<8} {dist_km:>8.1f} km")

print()
print("✅ Advantages of PostGIS:")
print("   - Circular search (not a box)")
print("   - Accurate distance on Earth's surface")
print("   - Sorted by distance (closest first)")
print("   - Uses spatial index → fast even with millions of rows")

## 🎯 Building the Tinder Feed: Location + Preferences

Finding nearby users is only half the story. Tinder also filters by:
1. **Gender preference** — show me only the gender(s) I'm interested in
2. **Age range** — show me only users within my preferred age range
3. **Mutual interest** — the other person should also be interested in my gender
4. **Already swiped** — don't show profiles I've already swiped on

Let's build the real feed query step by step.

### The Scenario
Let's generate the feed for **Emma Chen** (user 1):  
- She's 25, female, interested in males aged 22–32, within 15 km

In [ ]:
# First, let's see Emma's profile
emma = query("SELECT * FROM users WHERE id = 1")[0]

print("👤 Emma's Profile:")
print(f"   Name: {emma['name']}")
print(f"   Age: {emma['age']}")
print(f"   Gender: {emma['gender']}")
print(f"   Interested in: {emma['interested_in']}")
print(f"   Age range: {emma['age_min']} - {emma['age_max']}")
print(f"   Max distance: {emma['max_distance_km']} km")

In [ ]:
# Step 1: Find users within Emma's distance range who match her gender preference

step1 = query("""
    SELECT 
        u.id, u.name, u.age, u.gender, u.bio,
        ROUND(ST_Distance(u.location, me.location)::numeric, 0) AS distance_m
    FROM users u, users me
    WHERE me.id = 1              -- Emma
    AND u.id != me.id            -- not herself
    AND u.is_active = TRUE       -- only active users
    -- Location filter: within Emma's max distance
    AND ST_DWithin(u.location, me.location, me.max_distance_km * 1000)
    -- Gender filter: Emma wants males
    AND (me.interested_in = 'both' OR u.gender = me.interested_in)
    -- Age filter: within Emma's preferred range
    AND u.age BETWEEN me.age_min AND me.age_max
    -- Mutual interest: the other person should be interested in Emma's gender
    AND (u.interested_in = 'both' OR u.interested_in = me.gender)
    ORDER BY ST_Distance(u.location, me.location)
""")

print(f"📋 Step 1: Location + Preferences → {len(step1)} candidates\n")
print(f"{'Name':<20} {'Age':<5} {'Gender':<8} {'Bio':<45} {'Dist':>8}")
print("-" * 90)
for u in step1:
    dist_km = float(u['distance_m']) / 1000
    bio = (u['bio'][:42] + '...') if len(u['bio']) > 42 else u['bio']
    print(f"{u['name']:<20} {u['age']:<5} {u['gender']:<8} {bio:<45} {dist_km:>6.1f} km")

In [ ]:
# Step 2: Exclude users Emma has already swiped on

# First, let's see who Emma has already swiped on
already_swiped = query("""
    SELECT s.target_id, u.name, s.direction
    FROM swipes s
    JOIN users u ON u.id = s.target_id
    WHERE s.swiper_id = 1
""")

print("🔄 Emma has already swiped on:")
for s in already_swiped:
    emoji = "👍" if s['direction'] == 'right' else "👎"
    print(f"   {emoji} {s['name']} ({s['direction']})")
print()

In [ ]:
# The complete feed query — this is what Tinder's Profile Service runs

feed = query("""
    SELECT 
        u.id, u.name, u.age, u.gender, u.bio,
        ROUND(ST_Distance(u.location, me.location)::numeric, 0) AS distance_m
    FROM users u, users me
    WHERE me.id = 1              -- Emma
    AND u.id != me.id            -- not herself
    AND u.is_active = TRUE       -- only active users
    -- Location: within max distance
    AND ST_DWithin(u.location, me.location, me.max_distance_km * 1000)
    -- Gender preference (what Emma wants)
    AND (me.interested_in = 'both' OR u.gender = me.interested_in)
    -- Age preference (what Emma wants)
    AND u.age BETWEEN me.age_min AND me.age_max
    -- Mutual interest (other person wants Emma's gender too)
    AND (u.interested_in = 'both' OR u.interested_in = me.gender)
    -- Exclude already-swiped users
    AND NOT EXISTS (
        SELECT 1 FROM swipes s 
        WHERE s.swiper_id = me.id AND s.target_id = u.id
    )
    ORDER BY ST_Distance(u.location, me.location)
    LIMIT 20  -- stack size
""")

print(f"🃏 Emma's Swipe Stack: {len(feed)} profiles\n")
print(f"{'#':<3} {'Name':<20} {'Age':<5} {'Distance':>10}  Bio")
print("-" * 75)
for i, u in enumerate(feed, 1):
    dist_km = float(u['distance_m']) / 1000
    bio = (u['bio'][:35] + '...') if len(u['bio']) > 35 else u['bio']
    print(f"{i:<3} {u['name']:<20} {u['age']:<5} {dist_km:>8.1f} km  {bio}")

print()
print("💡 Notice: James Wilson is NOT in the feed — Emma already swiped right on him!")

## ⚡ Performance: Naive vs PostGIS

With 20 users, any approach is fast. But Tinder has **20 million daily active users**. Let's simulate a larger dataset and measure the difference.

We'll insert 10,000 fake users with random locations around LA, then compare a naive bounding box query vs a PostGIS spatial query.

In [ ]:
import random

# Insert 10,000 fake users scattered around Los Angeles
conn = get_db()
conn.autocommit = True
cursor = conn.cursor()

print("🔨 Inserting 10,000 test users...")

# LA bounding box: lat 33.7–34.3, lng -118.7 to -117.8
values = []
for i in range(10000):
    lat = random.uniform(33.7, 34.3)
    lng = random.uniform(-118.7, -117.8)
    age = random.randint(18, 45)
    gender = random.choice(['male', 'female'])
    interested_in = random.choice(['male', 'female', 'both'])
    values.append(f"('TestUser{i}', 'test{i}@test.com', {age}, '{gender}', 'Test bio', '{interested_in}', 18, 45, 50, ST_SetSRID(ST_MakePoint({lng}, {lat}), 4326))")

# Insert in one batch for speed
batch_sql = f"""
    INSERT INTO users (name, email, age, gender, bio, interested_in, age_min, age_max, max_distance_km, location)
    VALUES {','.join(values)}
"""
cursor.execute(batch_sql)
conn.close()

# Verify count
result = query("SELECT COUNT(*) as cnt FROM users")
print(f"✅ Database now has {result[0]['cnt']} users")

In [ ]:
# Benchmark: Naive bounding box vs PostGIS spatial query

my_location_sql = "ST_SetSRID(ST_MakePoint(-118.2437, 34.0522), 4326)"

# Naive approach: bounding box with manual distance calculation
naive_sql = """
    SELECT id, name, age, gender
    FROM users
    WHERE ST_Y(location::geometry) BETWEEN 34.0522 - 0.135 AND 34.0522 + 0.135
    AND ST_X(location::geometry) BETWEEN -118.2437 - 0.163 AND -118.2437 + 0.163
    AND gender = 'male'
    AND age BETWEEN 22 AND 32
    LIMIT 20
"""

# PostGIS approach: spatial index with ST_DWithin
postgis_sql = f"""
    SELECT id, name, age, gender
    FROM users
    WHERE ST_DWithin(location, {my_location_sql}, 15000)
    AND gender = 'male'
    AND age BETWEEN 22 AND 32
    ORDER BY ST_Distance(location, {my_location_sql})
    LIMIT 20
"""

# Benchmark each approach
def benchmark(sql, label, iterations=50):
    times = []
    for _ in range(iterations):
        conn = get_db()
        cursor = conn.cursor()
        start = time.time()
        cursor.execute(sql)
        cursor.fetchall()
        elapsed = (time.time() - start) * 1000
        times.append(elapsed)
        conn.close()
    avg = sum(times) / len(times)
    print(f"{label}")
    print(f"   Average: {avg:.2f} ms | Min: {min(times):.2f} ms | Max: {max(times):.2f} ms")
    return avg

print("⏱️  Benchmarking with 10,000+ users (50 iterations each):\n")
avg_naive = benchmark(naive_sql, "📦 Naive bounding box")
print()
avg_postgis = benchmark(postgis_sql, "🎯 PostGIS ST_DWithin")
print()

print(f"💡 PostGIS uses a spatial index (GIST) that organizes space into a")
print(f"   tree of bounding boxes, narrowing the search to a small region.")
print(f"   The naive approach must scan and filter every row in the range.")

In [ ]:
# Let's see the query plan to understand WHY PostGIS is faster

explain_results = query(f"""
    EXPLAIN ANALYZE
    SELECT id, name, age, gender
    FROM users
    WHERE ST_DWithin(location, {my_location_sql}, 15000)
    AND gender = 'male'
    AND age BETWEEN 22 AND 32
    ORDER BY ST_Distance(location, {my_location_sql})
    LIMIT 20
""")

print("📊 Query Plan (EXPLAIN ANALYZE):")
print("=" * 80)
for row in explain_results:
    # EXPLAIN returns results in a specific format
    print(list(row.values())[0])

print()
print("💡 Look for 'Index Scan using idx_users_location' — that's the spatial index!")
print("   Without it, PostgreSQL would do a sequential scan of ALL rows.")

## 🏗️ How Would This Work at Tinder Scale?

With 20M daily active users, even PostGIS queries on a single database would be too slow. Here's what Tinder does in production:

### 1. Pre-computed Feed Cache
A background job periodically generates a stack of candidate profiles for each user and stores it in **Redis**. When the user opens the app, they get this cached stack instantly (< 50ms).

### 2. Real-Time Fallback with Elasticsearch
If the user swipes through the cached stack, the system falls back to an **Elasticsearch** query (which has built-in geo queries) to generate more candidates in real-time.

### 3. The Hybrid Approach
```
User opens app → Check Redis for cached feed
  ├── Cache HIT → Return instantly (< 50ms)
  └── Cache MISS → Query Elasticsearch (< 300ms)
                    └── Background job: cache next batch
```

In this lab, we use PostGIS because it's the simplest way to understand geospatial queries. The concepts (spatial indexes, proximity search, preference filtering) are identical regardless of whether you use PostGIS, Elasticsearch, or a custom solution.

## 🧹 Cleanup

In [ ]:
# Remove the 10,000 test users we inserted
conn = get_db()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("DELETE FROM users WHERE name LIKE 'TestUser%'")
print(f"🧹 Removed {cursor.rowcount} test users")

result = query("SELECT COUNT(*) as cnt FROM users")
print(f"   Database back to {result[0]['cnt']} users")
conn.close()

## 📚 Summary

### Key Takeaways

1. **Regular SQL indexes fail for location queries** — they're one-dimensional (B-trees), but location is two-dimensional
2. **PostGIS adds spatial indexes** — GIST indexes organize space into a tree of bounding boxes for fast proximity search
3. **`ST_DWithin` is your friend** — finds all points within a radius, using the spatial index automatically
4. **The feed query combines location + preferences + already-swiped exclusion** — one query generates the swipe stack
5. **At scale, use pre-computed caches** — generate feeds in the background and serve from Redis/cache for instant load

### Next Up

In **Notebook 2**, we'll build the **Swipe and Match System** — recording swipes atomically with Redis Lua scripts and detecting mutual matches with strong consistency.